# Sparsity Induction

[Pruning](pruning.ipynb) removes weights from a model that was trained to use all of
them. **Sparsity induction** changes the training objective so the model never relies on
them in the first place — zeros come out of the optimiser, not out of a post-hoc mask.

The difference is not cosmetic. A densely-trained network distributes its function
across every weight it has; deleting some of them is genuinely destructive and you spend
a fine-tuning pass repairing the damage. A network trained under a sparsity-inducing
penalty converges to a solution that *is* sparse, so thresholding it costs almost
nothing.

Read [Pruning](pruning.ipynb) first — this notebook is the training-time counterpart and
refers to it throughout.

## 1. What & Why

The mechanism is a penalty added to the loss that makes non-zero weights *cost*
something:

```
L = task_loss(w)  +  λ · penalty(w)
```

Which penalty you choose decides whether you get true zeros or merely small numbers, and
that distinction is the whole subject. The classic result: **L1 produces exact zeros, L2
does not**, no matter how large you make λ.

**Reach for it when:**

- You own the training loop (this is its hard prerequisite — you cannot induce sparsity
  into someone else's finished checkpoint).
- You want higher sparsity than post-hoc pruning sustains. Induced sparsity typically
  holds accuracy several points of sparsity further.
- You want *structured* sparsity that is chosen by the optimiser rather than imposed by
  you — group lasso decides which channels to drop, so you do not have to.

**Don't when:**

- You have a checkpoint and no training budget. That is exactly what
  [pruning](pruning.ipynb) is for, and modern one-shot LLM pruners are very good.
- You need a specific hardware pattern like 2:4. You can train toward it, but it is
  easier to impose it and fine-tune.
- Your model is under-parameterised already. A penalty that removes capacity from a
  model that has none to spare just makes it worse.

## 2. Mental Model

**Corners.**

Constrained optimisation is a balloon being inflated inside a shape until it touches the
boundary. The loss contours are the balloon; the penalty defines the shape.

- The **L2** ball is a sphere. It is smooth everywhere, so the contact point is almost
  never exactly on an axis. Every coordinate ends up small but non-zero — L2 *shrinks*.
- The **L1** ball is a diamond, and diamonds have **corners that sit on the axes**. A
  corner is where one or more coordinates are exactly zero, and corners are where a
  balloon pushed against a diamond preferentially touches. L1 *selects*.

That single geometric fact — corners on the axes — is why lasso does feature selection
and ridge does not, and it is worth carrying around because it explains group lasso too:
group lasso's "ball" has corners along whole *subspaces*, so it zeroes out entire groups
of coordinates at once, which is how you get structured sparsity for free.

The other framing worth holding: pruning is **editing a finished essay down to the word
limit**; sparsity induction is **writing to the word budget from the first sentence**.
The second one reads better, and for the same reason.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **L1 / lasso** | Penalty `λ‖w‖₁`. Produces exact zeros — but only if optimised correctly (see the gotcha below). |
| **L2 / ridge / weight decay** | Penalty `λ‖w‖₂²`. Shrinks every weight toward zero, reaches zero for none of them. |
| **Proximal operator** | The closed-form "apply the penalty" step. For L1 it is **soft-thresholding**: `sign(w)·max(\|w\|−λη, 0)`. |
| **ISTA / proximal gradient** | Gradient step on the task loss, then a proximal step for the penalty. The standard way to actually get zeros from L1. |
| **Subgradient descent** | The naive alternative: differentiate `\|w\|` as `sign(w)`. Converges, but essentially never lands on exact zero. |
| **Group lasso** | Penalty `λ Σ_g ‖w_g‖₂` over predefined groups (a row, a channel, an attention head). Zeroes whole groups → **structured** sparsity. |
| **L0 regularisation** | Penalise the *count* of non-zeros directly. Non-differentiable, so it is done with stochastic gates (hard-concrete) and a straight-through estimator. |
| **Straight-through estimator (STE)** | Forward pass uses the hard/binary decision; backward pass pretends it was the identity. How you backprop through a mask. |
| **Movement pruning** | Importance = how far a weight moved *away from* zero during fine-tuning, not its magnitude. Learned, not fixed. |
| **Dynamic sparse training (RigL, SET)** | Keep a fixed sparsity budget throughout training, periodically dropping the weakest weights and **growing** new ones where gradients are largest. |
| **Sparse-to-sparse vs dense-to-sparse** | Whether the model is ever dense. Sparse-to-sparse saves training memory too, which is the point of RigL. |

## 4. Setup

NumPy only. Every method below is a handful of lines, and seeing the soft-threshold
operator produce exact zeros is more convincing than reading that it does.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — L1 gives exact zeros, L2 never does

A sparse-recovery problem: the true signal uses 8 of 60 features. Fit it three ways and
count how many coefficients are *exactly* `0.0`.

In [2]:
n, d, k = 200, 60, 8
w_true = np.zeros(d)
support = rng.choice(d, size=k, replace=False)
w_true[support] = rng.standard_normal(k) * 2.0

X = rng.standard_normal((n, d))
y = X @ w_true + rng.standard_normal(n) * 0.3

def soft_threshold(w, t):
    '''The proximal operator of the L1 norm -- this is what creates exact zeros.'''
    return np.sign(w) * np.maximum(np.abs(w) - t, 0.0)

def fit(X, y, penalty, lam, steps=4000, lr=0.002):
    w = np.zeros(X.shape[1])
    for _ in range(steps):
        grad = X.T @ (X @ w - y) / len(y)
        if penalty == "l2":
            w -= lr * (grad + 2 * lam * w)             # smooth, differentiable
        elif penalty == "l1_subgradient":
            w -= lr * (grad + lam * np.sign(w))        # the naive way
        elif penalty == "l1_proximal":
            w = soft_threshold(w - lr * grad, lr * lam)  # ISTA
    return w

fits = {
    "L2 (ridge), lam=0.5":        fit(X, y, "l2", 0.5),
    "L1 subgradient, lam=0.5":    fit(X, y, "l1_subgradient", 0.5),
    "L1 proximal (ISTA), lam=0.5": fit(X, y, "l1_proximal", 0.5),
}

print(f"true model uses {k} of {d} features\n")
print(f"{'method':30} {'exact zeros':>12} {'|w|<1e-3':>10} {'test err':>10}")
X_te = rng.standard_normal((2000, d)); y_te = X_te @ w_true
for name, w in fits.items():
    err = np.linalg.norm(X_te @ w - y_te) / np.linalg.norm(y_te)
    print(f"{name:30} {int((w == 0).sum()):12d} {int((np.abs(w) < 1e-3).sum()):10d} {err:10.4f}")

print("\nRidge drives everything small and nothing to zero.")
print("L1 by subgradient descent *also* produces no exact zeros -- it oscillates across")
print("the kink at 0 forever. Only the proximal step lands on it.")

true model uses 8 of 60 features

method                          exact zeros   |w|<1e-3   test err
L2 (ridge), lam=0.5                       0          1     0.5185
L1 subgradient, lam=0.5                   0         49     0.1861
L1 proximal (ISTA), lam=0.5              54         54     0.1860

Ridge drives everything small and nothing to zero.
L1 by subgradient descent *also* produces no exact zeros -- it oscillates across
the kink at 0 forever. Only the proximal step lands on it.


That middle row is the practical trap. "I added an L1 penalty and my model isn't sparse"
is almost always this: the penalty was correct and the *optimiser* could not represent
the answer. L1 needs a proximal step (or an explicit threshold at the end); adding
`lam * sign(w)` to the gradient is not enough.

### Example 2 — the sparsity/accuracy frontier

Sweep λ and watch the model trade features for error. This curve is the thing you
actually tune against.

In [3]:
print(f"{'lambda':>8} {'nonzeros':>9} {'support found':>14} {'false pos':>10} {'test err':>10}")
for lam in (0.01, 0.05, 0.2, 0.5, 1.0, 2.0, 5.0):
    w = fit(X, y, "l1_proximal", lam)
    nz = np.flatnonzero(w)
    found = len(set(nz) & set(support))
    err = np.linalg.norm(X_te @ w - y_te) / np.linalg.norm(y_te)
    print(f"{lam:8.2f} {len(nz):9d} {f'{found}/{k}':>14} {len(nz) - found:10d} {err:10.4f}")

print("\nToo small a lambda keeps every feature; too large starts deleting real ones.")
print("The best test error is NOT at zero penalty -- sparsity is also regularisation.")

  lambda  nonzeros  support found  false pos   test err
    0.01        44            8/8         36     0.0186
    0.05        10            8/8          2     0.0193


    0.20         7            7/8          0     0.0759
    0.50         6            6/8          0     0.1860
    1.00         6            6/8          0     0.3562


    2.00         4            4/8          0     0.6248


    5.00         1            1/8          0     0.9796

Too small a lambda keeps every feature; too large starts deleting real ones.
The best test error is NOT at zero penalty -- sparsity is also regularisation.


### Example 3 — the head-to-head: induce sparsity, or train dense and prune?

This is the comparison that justifies the technique, and it needs the right regime to be
meaningful. With plenty of data per parameter, an ordinary fit followed by
[magnitude pruning](pruning.ipynb) does fine — the coefficients are well estimated and
the big ones really are the important ones.

The interesting case is the one every modern network is in: **more parameters than
data**. Here a dense fit is under-determined, its coefficients are arbitrary, and
ranking them by magnitude is close to meaningless.

In [4]:
# The over-parameterised regime: 200 features, 60 observations, 8 of them real.
n2, d2, k2 = 60, 200, 8
w_true2 = np.zeros(d2)
support2 = rng.choice(d2, size=k2, replace=False)
w_true2[support2] = rng.standard_normal(k2) * 2.0

X2 = rng.standard_normal((n2, d2))
y2 = X2 @ w_true2 + rng.standard_normal(n2) * 0.3
X2_te = rng.standard_normal((3000, d2)); y2_te = X2_te @ w_true2

step = 1.0 / (np.linalg.norm(X2, 2) ** 2 / n2)      # 1/L, the safe proximal step size

def keep_top_k(w, k):
    out = np.zeros_like(w)
    idx = np.argsort(-np.abs(w))[:k]
    out[idx] = w[idx]
    return out

def refit_on_support(X, y, w, steps=3000, lr=step):
    '''Retrain only the surviving coefficients -- 'fine-tune after pruning', and also
    the standard debiasing step after a lasso fit.'''
    mask, v = (w != 0), w.copy()
    for _ in range(steps):
        v -= lr * (X.T @ (X @ v - y) / len(y))
        v *= mask                                   # pruned weights stay at zero
    return v

dense2 = fit(X2, y2, "l2", 0.01, lr=step)           # ordinary training, no sparsity pressure
l1_2 = fit(X2, y2, "l1_proximal", 0.05, lr=step)    # trained toward sparsity

err2 = lambda w: np.linalg.norm(X2_te @ w - y2_te) / np.linalg.norm(y2_te)

print(f"{'budget':>7} | {'dense -> prune':>14} {'+ refit':>9} | {'L1 -> prune':>12} "
      f"{'+ refit':>9} {'support':>9}")
for budget in (8, 16, 32):
    a = keep_top_k(dense2, budget)
    b = keep_top_k(l1_2, budget)
    hits = len(set(np.flatnonzero(b)) & set(support2))
    print(f"{budget:7d} | {err2(a):14.4f} {err2(refit_on_support(X2, y2, a)):9.4f} | "
          f"{err2(b):12.4f} {err2(refit_on_support(X2, y2, b)):9.4f} {f'{hits}/{k2}':>9}")

print("\nAn order of magnitude apart, at identical final sparsity. The dense fit never")
print("knew which features mattered, so magnitude pruning was ranking noise. The")
print("L1-trained model identified the *support* during training -- which is the thing")
print("that cannot be recovered afterwards.")

 budget | dense -> prune   + refit |  L1 -> prune   + refit   support


      8 |         0.7549    0.3904 |       0.0876    0.0492       7/8
     16 |         0.7642    0.3757 |       0.0964    0.1232       7/8
     32 |         0.7712    0.1182 |       0.0967    0.1213       7/8

An order of magnitude apart, at identical final sparsity. The dense fit never
knew which features mattered, so magnitude pruning was ranking noise. The
L1-trained model identified the *support* during training -- which is the thing
that cannot be recovered afterwards.


The `+ refit` columns are worth reading closely, because they do **not** behave the same
way on both sides.

On the pruning side refitting always helps a lot — it is the recovery fine-tune the
pruning literature insists on, and it is doing most of that method's work.

On the lasso side it helps at the smallest budget (where the support is roughly the
right size) and actively *hurts* at 16 and 32. That is not a bug: refitting removes L1's
shrinkage, and shrinkage was the only thing keeping the false positives harmless. Once
you refit an over-large support with 60 observations, those spurious coefficients are
free to overfit.

So the rule is narrower than "always debias": use the penalty to choose *which* weights,
refit to decide *how large* — but only when the selected support is genuinely small
relative to your data. Otherwise leave the shrinkage in place.

### Example 4 — group lasso: structured sparsity, chosen by the optimiser

[Pruning](pruning.ipynb) Example 3 showed that only *structured* zeros make a matmul
faster. Group lasso induces exactly that: penalise each group's L2 norm and whole groups
collapse together.

The comparison has to be at **matched sparsity** to mean anything, so λ is searched for
each method to hit the same non-zero count.

In [5]:
# Treat 60 features as 12 groups of 5 -- think "5 weights belonging to one channel".
G, gsize = 12, 5
groups = np.arange(d).reshape(G, gsize)

w_grouped = np.zeros(d)
active_groups = rng.choice(G, size=2, replace=False)
for g in active_groups:
    w_grouped[groups[g]] = rng.standard_normal(gsize) * 2.0
y_g = X @ w_grouped + rng.standard_normal(n) * 0.3
step_g = 1.0 / (np.linalg.norm(X, 2) ** 2 / n)

def block_soft_threshold(w, t):
    '''Proximal operator of the group-L2 norm: shrink each group's *norm* toward zero,
    so the whole group crosses zero together.'''
    out = w.copy()
    for idx in groups:
        norm = np.linalg.norm(w[idx])
        out[idx] = w[idx] * max(0.0, 1.0 - t / norm) if norm > 0 else 0.0
    return out

def fit_group(lam, steps=4000):
    w = np.zeros(d)
    for _ in range(steps):
        w = block_soft_threshold(w - step_g * (X.T @ (X @ w - y_g) / n), step_g * lam)
    return w

def fit_l1_g(lam, steps=4000):
    w = np.zeros(d)
    for _ in range(steps):
        w = soft_threshold(w - step_g * (X.T @ (X @ w - y_g) / n), step_g * lam)
    return w

def match_sparsity(fitfn, target):
    '''Search lambda for the fit closest to `target` non-zeros.'''
    best = None
    for lam in np.logspace(-3, 1, 60):
        w = fitfn(lam)
        nnz = int((w != 0).sum())
        if best is None or abs(nnz - target) < abs(best[1] - target):
            best = (lam, nnz, w)
    return best

dead_groups = lambda w: sum(1 for idx in groups if not np.any(w[idx]))

print(f"true model: {len(active_groups)} of {G} groups active ({len(active_groups)*gsize} non-zeros)\n")
print(f"{'budget':>7} {'method':>17} {'nonzeros':>9} {'dead groups':>12} {'params removable':>17}")
for target in (10, 15, 20):
    for name, fn in (("element-wise L1", fit_l1_g), ("group lasso", fit_group)):
        lam, nnz, w = match_sparsity(fn, target)
        print(f"{target:7d} {name:>17} {nnz:9d} {dead_groups(w):9d}/{G} {dead_groups(w)*gsize:17d}")

print("\nAt the very sparsest setting the two nearly tie -- squeeze hard enough and")
print("element-wise L1 concentrates on its own. The gap opens as soon as you back off:")
print("at ~20 non-zeros, L1's survivors are scattered across most groups so barely a")
print("channel can be deleted, while group lasso is still emptying whole channels.")
print("\nSame sparsity, very different *shape* -- and shape is what buys latency.")

true model: 2 of 12 groups active (10 non-zeros)

 budget            method  nonzeros  dead groups  params removable


     10   element-wise L1        10        10/12                50


     10       group lasso        10        10/12                50


     15   element-wise L1        14         6/12                30


     15       group lasso        15         9/12                45


     20   element-wise L1        21         3/12                15


     20       group lasso        25         7/12                35

At the very sparsest setting the two nearly tie -- squeeze hard enough and
element-wise L1 concentrates on its own. The gap opens as soon as you back off:
at ~20 non-zeros, L1's survivors are scattered across most groups so barely a
channel can be deleted, while group lasso is still emptying whole channels.

Same sparsity, very different *shape* -- and shape is what buys latency.


## 6. Gotchas & Pitfalls

- **Subgradient descent on L1 does not give zeros.** Example 1's middle row. Use a
  proximal step, or accept that you must threshold at the end (in which case you are
  really doing pruning with an L1-shaped prior).
- **`weight_decay` in AdamW is not L1.** It is decoupled **L2**. Setting it high gives
  you a uniformly small, entirely dense model. There is no L1 option in the standard
  optimisers; you add it yourself.
- **Penalising things that should not be penalised.** Biases, LayerNorm/BatchNorm gains
  and embeddings should normally be excluded. Shrinking a normalisation scale toward
  zero attacks the network's conditioning, not its redundancy.
- **A fixed λ from step 0.** Full penalty on an untrained model suppresses weights before
  they have learned anything, and the model never recovers the support. Warm up λ, or
  train dense for a while first.
- **λ is not transferable.** It scales with the loss magnitude, the batch size and the
  number of parameters. A λ tuned on one layer size is meaningless on another; tune
  against a *target sparsity*, not a target λ.
- **Unstructured induced sparsity still does not speed anything up.** Everything in
  [Pruning](pruning.ipynb) about dense kernels applies identically here. If you want
  latency, use group lasso (or an N:M-aware objective), not element-wise L1.
- **Dynamic sparse training without the grow step.** RigL's contribution is *regrowth* —
  drop-only schedules monotonically lose capacity and cannot recover from an early bad
  drop.
- **Reporting "≈0" as sparse.** Weights of `1e-7` are dense weights. Until they are
  exactly zero and stored as such, you have saved nothing.

## 7. When to Use vs Alternatives

| Situation | Best tool |
|---|---|
| You have a checkpoint, no training budget | [Pruning](pruning.ipynb) — Wanda/SparseGPT one-shot |
| You own training, want max sparsity | **Sparsity induction** (this notebook) |
| You want a smaller *shape* | **Group lasso**, or [structured pruning](pruning.ipynb), or [distillation](knowledge-distillation.ipynb) |
| You want the cheapest real win | [Quantization](quantization-gptq-awq.ipynb) |
| Training memory is the binding constraint | **Dynamic sparse training** (RigL) — never materialises a dense model |
| Fine-tuning a pretrained model sparsely | **Movement pruning** — magnitude is the wrong criterion under transfer |

**The honest position.** For large language models, post-hoc methods currently dominate
in practice, for a boring reason: nobody wants to redo a foundation-model pretraining run
to get sparsity, and one-shot pruners have become very good. Sparsity induction earns its
place when you are training the model anyway — a domain model, a fine-tune, a small
network destined for an edge device — and especially when you use group lasso to let the
optimiser choose the architecture.

The exception worth knowing is **dynamic sparse training**, which is the only technique
here that reduces the cost of *training* rather than inference, because the model is
never dense at any point.

## 8. Resources

- [Regression Shrinkage and Selection via the Lasso](https://www.jstor.org/stable/2346178) — Tibshirani, 1996. The origin of the corners argument.
- [Model Compression via Distillation and Quantization](https://arxiv.org/abs/1802.05668) — a useful counterpoint on combining the axes.
- [Learning Sparse Neural Networks through L0 Regularization](https://arxiv.org/abs/1712.01312) — Louizos et al.; hard-concrete gates and the straight-through trick.
- [Movement Pruning: Adaptive Sparsity by Fine-Tuning](https://arxiv.org/abs/2005.07683) — why magnitude fails during transfer learning, and what to use instead.
- [Rigging the Lottery: Making All Tickets Winners](https://arxiv.org/abs/1911.11134) — RigL; sparse-to-sparse training with prune-and-grow.
- [Learning Structured Sparsity in Deep Neural Networks](https://arxiv.org/abs/1608.03665) — group lasso applied to channels and filters, the basis of Example 4.
- [Proximal Algorithms](https://web.stanford.edu/~boyd/papers/prox_algs.html) — Parikh & Boyd; the reference for proximal operators, including every one used here.